# 04 · Ajuste fino con RECORTES de panel (CNN y transformer), validación por artículo

**Dónde correrlo:** Jupyter en `nukwa-l40s` (o `nukwa-v100`), **1 nodo, 4 núcleos, 16 GB, 2 h**. Kernel `Python-ML-DL` (PyTorch con CUDA, timm).

**Antes de empezar (una vez):** sube `recortes_v02.zip` (≈ 400 MB) a `~/ondemand/Kabre_Nano2Predict/` (o a su subcarpeta `datos/`); la celda 1 lo descomprime sola la primera vez (deja `recortes/<pmcid>/*.png` y `datos/indice_recortes_entrenamiento.csv`). Si el cuaderno 03 ya corrió, `resultados/03_ajuste_fino/predicciones_ft_*.csv` sirven para la comparación final.

**Qué cambia respecto al 03:**
1. **Datos:** cada muestra es un **recorte de un panel de micrografía de partículas** (separado de su figura compuesta), no la figura completa. Solo recortes `elegible_morfologia` (TEM/SEM/AFM/cryo de partículas, sin bacterias, células ni tejido) con clase de portador y fold por artículo. Un artículo nunca está en dos particiones.
2. **Regularización:** `drop_path_rate = 0,1` (profundidad estocástica) y recorte de aumento más suave (escala 0,7–1,0, para no perder la barra de escala).
3. **Evaluación por recorte y por artículo:** el artículo reúne varios recortes; se promedian las probabilidades por artículo y se compara con el cuaderno 03 **solo en los artículos comunes**, para que la comparación sea justa.
4. **Variante B** (opcional): sin las figuras con contenido biológico, si existe `datos/excluir_bio.csv` (columna `fig_id`).

**Misma receta que el 03:** AdamW con tasa de aprendizaje por capas, OneCycleLR, precisión mixta, pesos de clase suavizados, label smoothing 0,1, aumentos en la GPU, **punto de control por época** (reanudable) y número de épocas fijo: **no se elige nada con los datos de prueba**.

**Tiempo:** la celda 3 mide cuánto tardará (≈ 1 min) y estima el total antes de gastar la sesión. El 03 tardaba 2–4 min por partición y modelo con ≈ 2.100 imágenes; aquí hay ≈ 2,2 veces más muestras, o sea unos 4–9 min por partición y modelo (cálculo, no medida). La celda 4 se detiene sola antes del límite (`TIEMPO_MAX_MIN`).

**Referencias:** exactitud 0,485 / F1 macro 0,466 (vectores congelados, figura completa) y el resultado del 03 (figura completa, ajuste fino). Clase mayoritaria: se imprime en la celda 1.


In [ ]:
# Celda 1: GPU, datos (recortes) y carga en memoria
import os, sys, json, time, math, random, socket, platform
from pathlib import Path
BASE = Path(os.environ.get("N2P_BASE", Path.home() / "ondemand" / "Kabre_Nano2Predict"))
DATOS = BASE / "datos"
T_INICIO = time.time()

import numpy as np, pandas as pd, torch, timm
import torch.nn as nn, torch.nn.functional as F
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
assert torch.cuda.is_available(), "No hay GPU: pide una cola nukwa-* y usa el kernel Python-ML-DL"
dev = "cuda"
asignados = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("nodo:", socket.gethostname(), "| GPU:", torch.cuda.get_device_name(0), "| núcleos asignados:", asignados)
print("torch", torch.__version__, "| timm", timm.__version__)

# --- datos: recortes de panel ---
ZIP = next((z for z in (BASE / "recortes_v02.zip", DATOS / "recortes_v02.zip") if z.exists()), BASE / "recortes_v02.zip")
if ZIP.exists() and not (BASE / "recortes").exists():          # primera vez: descomprime solo
    import zipfile; DATOS.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z: z.extractall(BASE)
    (BASE / "indice_recortes_entrenamiento.csv").replace(DATOS / "indice_recortes_entrenamiento.csv")
    print("recortes descomprimidos en", BASE / "recortes")
VARIANTE = "A_todos"          # "A_todos" | "B_sin_biologico" (necesita datos/excluir_bio.csv con la columna fig_id)
REC = BASE / "recortes"
SALIDA = BASE / "resultados" / "04_recortes" / VARIANTE
SALIDA.mkdir(parents=True, exist_ok=True)
t = pd.read_csv(DATOS / "indice_recortes_entrenamiento.csv", encoding="utf-8-sig", dtype=str)
if VARIANTE.startswith("B"):
    ex = pd.read_csv(DATOS / "excluir_bio.csv", encoding="utf-8-sig", dtype=str)
    antes = len(t); t = t[~t.fig_id.isin(set(ex.fig_id))]
    print(f"variante B: se excluyen {antes - len(t)} recortes de figuras con contenido biológico")
t["ruta"] = t.apply(lambda r: REC / str(r["pmcid"]) / str(r["image_file"]), axis=1)
t["existe"] = t["ruta"].map(lambda p: p.exists())
print("recortes en el índice:", len(t), "| que existen en disco:", int(t.existe.sum()))
d = t[t.existe & t.clase_modelo.notna()].reset_index(drop=True)
d["fold"] = d.fold.astype(int)
print("recortes usables:", len(d), "| artículos:", d.pmcid.nunique())
assert len(d) > 1000, "faltan recortes: sube y descomprime recortes_v02.zip en la carpeta base"
assert (d.groupby("pmcid").fold.nunique() == 1).all(), "un artículo está en dos folds"
clases = sorted(set(d.clase_modelo)); cidx = {c: i for i, c in enumerate(clases)}
yi = np.array([cidx[c] for c in d.clase_modelo]); folds = d.fold.astype(int).values
base = pd.Series(d.clase_modelo).value_counts(normalize=True).max()
print("clases:", clases, "| línea base (mayoritaria):", round(float(base), 3))

LADO = 256
def cargar(ruta):
    im = Image.open(ruta).convert("RGB"); w, h = im.size; l = max(w, h)
    lienzo = Image.new("RGB", (l, l), (255, 255, 255)); lienzo.paste(im, ((l - w) // 2, (l - h) // 2))   # recorte, rellenado a cuadrado
    return np.asarray(lienzo.resize((LADO, LADO), Image.BICUBIC), dtype=np.uint8)
t0 = time.time()
with ThreadPoolExecutor(max_workers=max(2, asignados)) as ex:
    X = np.stack(list(ex.map(cargar, [str(p) for p in d.ruta])))
Xg = torch.from_numpy(X).permute(0, 3, 1, 2).contiguous().to(dev)       # uint8 en la GPU (~0,4 GB)
del X
print(f"imágenes en la GPU: {tuple(Xg.shape)} ({time.time()-t0:.0f}s)")
print("OK celda 1")


In [ ]:
# Celda 2: aumentos en la GPU, modelo y bucle de entrenamiento con punto de control por época
TAM = 224
DROP_PATH = 0.1                 # regularización nueva respecto al 03
MEDIA = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1) * 255
DESV = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1) * 255

def lote_train(idx):
    xb, out = Xg[idx], []
    for i in range(xb.shape[0]):
        s = random.uniform(0.7, 1.0); c = int(LADO * math.sqrt(s))
        x0, y0 = random.randint(0, LADO - c), random.randint(0, LADO - c)
        im = F.interpolate(xb[i:i + 1, :, y0:y0 + c, x0:x0 + c].float(), size=(TAM, TAM), mode="bilinear", align_corners=False)
        out.append(im.flip(3) if random.random() < 0.5 else im)
    return (torch.cat(out) - MEDIA) / DESV

def lote_eval(idx):
    return (F.interpolate(Xg[idx].float(), size=(TAM, TAM), mode="bilinear", align_corners=False) - MEDIA) / DESV

MODELOS = {"resnet50": "resnet50.a1_in1k", "dinov2_vits14": "vit_small_patch14_dinov2.lvd142m"}
os.environ.setdefault("HF_HOME", str(BASE / "hf_cache"))

def crear(corto):
    kw = {"img_size": TAM} if "vit" in MODELOS[corto] else {}
    kw["drop_path_rate"] = DROP_PATH
    return timm.create_model(MODELOS[corto], pretrained=True, num_classes=len(clases), **kw).to(dev)

def entrenar_fold(corto, k, epocas, lr, bs=32, semilla=0):
    tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
    random.seed(semilla + k); np.random.seed(semilla + k); torch.manual_seed(semilla + k)
    m = crear(corto)
    ids_cab = {id(p) for p in m.get_classifier().parameters()}
    bb = [p for p in m.parameters() if id(p) not in ids_cab]; cab = [p for p in m.parameters() if id(p) in ids_cab]
    opt = torch.optim.AdamW([{"params": bb, "lr": lr}, {"params": cab, "lr": lr * 10}], weight_decay=0.05)
    pasos_ep = len(tr) // bs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr, lr * 10], total_steps=epocas * pasos_ep, pct_start=0.15)
    frec = np.bincount(yi[tr], minlength=len(clases))
    w = torch.tensor((frec.sum() / (len(clases) * frec)) ** 0.5, dtype=torch.float32, device=dev)   # pesos por clase suavizados
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler()
    ytr = torch.tensor(yi, device=dev)
    ckpt = SALIDA / f"ckpt_{corto}_f{k}.pt"
    ep0 = 0
    if ckpt.exists():                                        # reanudar si se cortó
        s = torch.load(ckpt, map_location=dev)
        m.load_state_dict(s["m"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"]); ep0 = s["ep"] + 1
        print(f"  reanudando {corto} fold {k} desde la época {ep0}")
    for ep in range(ep0, epocas):
        m.train(); perm = np.random.permutation(tr); perd = ok = n = 0
        for i in range(pasos_ep):
            idx = torch.tensor(perm[i * bs:(i + 1) * bs], device=dev)
            with torch.autocast("cuda", dtype=torch.float16):
                lo = m(lote_train(idx)); loss = crit(lo, ytr[idx])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            perd += float(loss) * len(idx); ok += int((lo.argmax(1) == ytr[idx]).sum()); n += len(idx)
        print(f"  {corto} fold {k} época {ep+1}/{epocas}: pérdida {perd/n:.3f} | acierto de entrenamiento {ok/n:.3f} | {time.time()-T_INICIO:.0f}s", flush=True)
        torch.save({"m": m.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(), "ep": ep}, ckpt)   # punto de control
    m.eval(); P = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(te), 128):
            P.append(torch.softmax(m(lote_eval(torch.tensor(te[i:i + 128], device=dev))).float(), 1).cpu().numpy())
    P = np.concatenate(P)
    df = pd.DataFrame({"pmcid": d.pmcid.values[te], "image_file": d.image_file.values[te], "real": d.clase_modelo.values[te],
                       "prediccion": [clases[j] for j in P.argmax(1)], "fold": k})
    for j, c in enumerate(clases):
        df["p_" + c] = P[:, j].round(5)
    df.to_csv(SALIDA / f"ft_{corto}_fold{k}.csv", index=False, encoding="utf-8")
    ckpt.unlink(missing_ok=True)                              # libera disco (cuota de 10 GB)
    del m, opt; torch.cuda.empty_cache()
    return df
print("OK celda 2")


In [ ]:
# Celda 3: PRUEBA MÍNIMA (1 minuto): comprueba que cada modelo entrena en la GPU y estima cuánto tardará
torch.backends.cudnn.benchmark = True
def prueba_minima(corto, lotes=6, bs=32):
    m = crear(corto); m.train()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-5); scaler = torch.cuda.amp.GradScaler()
    ytr = torch.tensor(yi, device=dev); tiempos = []
    for i in range(lotes):
        idx = torch.randint(0, len(d), (bs,), device=dev)
        torch.cuda.synchronize(); t1 = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            lo = m(lote_train(idx)); loss = F.cross_entropy(lo, ytr[idx])
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        torch.cuda.synchronize(); tiempos.append(time.time() - t1)
        assert torch.isfinite(loss), "pérdida no finita"
    seg_lote = float(np.median(tiempos[2:]))                       # se descartan los primeros lotes (calentamiento)
    pasos_ep = int(len(d) * 0.8) // bs
    print(f"{corto}: salida {tuple(lo.shape)} | {seg_lote*1000:.0f} ms por lote | ≈ {seg_lote*pasos_ep:.0f} s por época | ≈ {seg_lote*pasos_ep*10/60:.1f} min por partición (10 épocas) | memoria GPU máx {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
    del m, opt; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    return seg_lote * pasos_ep * 10 / 60
est = {c: prueba_minima(c) for c in MODELOS}
print(f"\nEstimación total para 5 particiones y {len(MODELOS)} modelos: ≈ {sum(est.values())*5:.0f} min")
print("OK celda 3 (si ves esto, el entrenamiento funciona; sigue con la celda 4)")


In [ ]:
# Celda 4: correr los modelos y las particiones (reanudable; se detiene antes del límite de tiempo)
MODELOS_A_CORRER = ["resnet50", "dinov2_vits14"]
FOLDS_A_CORRER = [0, 1, 2, 3, 4]       # para una prueba rápida: [0]
EPOCAS = 10
LR = {"resnet50": 1e-4, "dinov2_vits14": 3e-5}
TIEMPO_MAX_MIN = 100                     # para una sesión de 2 h; usa 50 si pides 1 h

parar = False
for corto in MODELOS_A_CORRER:
    for k in FOLDS_A_CORRER:
        if (SALIDA / f"ft_{corto}_fold{k}.csv").exists():
            print(f"{corto} fold {k}: ya hecho"); continue
        if (time.time() - T_INICIO) / 60 > TIEMPO_MAX_MIN:
            print(f"\nLímite de tiempo ({TIEMPO_MAX_MIN} min). Vuelve a ejecutar esta celda en otra sesión: retoma donde quedó."); parar = True; break
        t1 = time.time()
        df = entrenar_fold(corto, k, EPOCAS, LR[corto])
        acc = (df.real == df.prediccion).mean()
        print(f"== {corto} fold {k} listo en {time.time()-t1:.0f}s | exactitud en la partición de prueba: {acc:.3f}\n", flush=True)
    if parar:
        break
print("OK celda 4")


In [ ]:
# Celda 5: métricas por recorte y por artículo, y comparación con el cuaderno 03 en los artículos comunes
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
pc = ["p_" + c for c in clases]
def por_articulo(o):
    g = o.groupby("pmcid").agg({**{c: "mean" for c in pc}, "real": "first", "fold": "first"})
    g["prediccion"] = [clases[j] for j in g[pc].values.argmax(1)]
    return g
def metr(real, pred):
    return {"accuracy": round(float(accuracy_score(real, pred)), 4), "accuracy_balanceada": round(float(balanced_accuracy_score(real, pred)), 4), "f1_macro": round(float(f1_score(real, pred, average="macro")), 4)}
resumen = {"variante": VARIANTE, "evaluacion": "5 particiones por articulo; ultima epoca; sin seleccion con datos de prueba", "clases": clases, "epocas": EPOCAS, "drop_path": DROP_PATH,
           "linea_base_clase_mayoritaria_recortes": round(float(base), 4), "modelos": {}}
por_art = {}
for corto in MODELOS:
    fs = [SALIDA / f"ft_{corto}_fold{k}.csv" for k in range(5)]
    if not all(f.exists() for f in fs):
        print(corto, "-> faltan particiones:", [f.name for f in fs if not f.exists()]); continue
    o = pd.concat([pd.read_csv(f) for f in fs]).sort_values(["pmcid", "image_file"]).reset_index(drop=True)
    o.to_csv(SALIDA / f"predicciones_ft_{corto}.csv", index=False, encoding="utf-8")
    art = por_articulo(o); por_art[corto] = art
    r = {"por_recorte": {**metr(o.real, o.prediccion), "n": int(len(o))}, "por_articulo": {**metr(art.real, art.prediccion), "n": int(len(art))},
         "matriz_confusion_recorte": confusion_matrix(o.real, o.prediccion, labels=clases).tolist()}
    ref = BASE / "resultados" / "03_ajuste_fino" / f"predicciones_ft_{corto}.csv"
    if ref.exists():
        a3 = por_articulo(pd.read_csv(ref)); com = art.index.intersection(a3.index)
        r["comparacion_con_03_articulos_comunes"] = {"n_articulos": int(len(com)), "recortes": metr(art.loc[com].real, art.loc[com].prediccion), "figura_completa_03": metr(a3.loc[com].real, a3.loc[com].prediccion)}
    resumen["modelos"][corto] = r
    print(corto, "| por recorte:", r["por_recorte"], "\n  por artículo:", r["por_articulo"])
    if "comparacion_con_03_articulos_comunes" in r: print("  frente al 03 (mismos artículos):", r["comparacion_con_03_articulos_comunes"])
    print(pd.DataFrame(r["matriz_confusion_recorte"], index=clases, columns=clases).to_string(), "\n")
if len(por_art) == 2:
    a, b = por_art["resnet50"], por_art["dinov2_vits14"]; com = a.index.intersection(b.index)
    pm = (a.loc[com, pc].values + b.loc[com, pc].values) / 2
    resumen["modelos"]["promedio_resnet_dinov2_por_articulo"] = metr(a.loc[com].real, [clases[j] for j in pm.argmax(1)])
    print("promedio de los dos modelos (por artículo):", resumen["modelos"]["promedio_resnet_dinov2_por_articulo"])
(SALIDA / "metricas_04.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nArchivos para descargar en", SALIDA, ":")
for p in sorted(SALIDA.glob("*")): print(" ", p.name, f"{p.stat().st_size/1e6:.1f} MB")
print("OK celda 5. Descarga metricas_04.json y predicciones_ft_*.csv y pégaselos a Claude.")
